# LLM Fiscal and Monetary Policy Classification Demo

Demonstrates fiscal and monetary policy classification prompts with 4 variants each: simple, with_definitions, chain_of_thought, and few_shot.

In [24]:
# Setup
from dotenv import load_dotenv
load_dotenv('../../.env')
import os, sys
sys.path.insert(0, '../../libs')
sys.path.insert(0, '../../src/Traction/prompts')

from llm_factory_openai import LLMAgent
from prompt_utils import load_prompt, format_messages
from schema import PROMPT_REGISTRY
from pathlib import Path

#### Set up llm

In [15]:
# Initialize LLM Agent
llm_agent = LLMAgent(
    api_key=os.getenv('OPENAI_API_KEY'),
    model='gpt-5-nano',
    temperature=1
)

#### Load prompts templates

In [16]:
# Group prompts by category
prompts = {'monetary_stance': [], 'monetary_agreement': [], 'fiscal_stance': [], 'fiscal_agreement': []}
for key, entry in PROMPT_REGISTRY.items():
    fname = entry["prompt_file"]
    for category in prompts.keys():
        if fname.startswith(category):
            prompts[category].append(key)
            break

print(f"Loaded {sum(len(v) for v in prompts.values())} prompts across {len(prompts)} categories")

Loaded 16 prompts across 4 categories


In [30]:
prompts

{'monetary_stance': ['monetary_stance_simple',
  'monetary_stance_with_definitions',
  'monetary_stance_few_shot',
  'monetary_stance_chain_of_thought'],
 'monetary_agreement': ['monetary_agreement_simple',
  'monetary_agreement_with_definitions',
  'monetary_agreement_few_shot',
  'monetary_agreement_chain_of_thought'],
 'fiscal_stance': ['fiscal_stance_simple',
  'fiscal_stance_with_definitions',
  'fiscal_stance_few_shot',
  'fiscal_stance_chain_of_thought'],
 'fiscal_agreement': ['fiscal_agreement_simple',
  'fiscal_agreement_with_definitions',
  'fiscal_agreement_few_shot',
  'fiscal_agreement_chain_of_thought']}

In [18]:
# Sample data
monetary_sample = {
    'text': "The accommodative monetary policy is appropriate given current economic conditions, but the central bank should gradually move towards a neutral stance in 2024.",
    'country': 'Brazil', 'year': '2024', 'text_author': 'IMF staff'
}

monetary_agreement_sample = {
    'staff': "The accommodative monetary policy is appropriate, but should gradually move towards a neutral stance in 2024.",
    'authority': "Monetary policy will remain broadly accommodative to support growth. The central bank will monitor conditions before adjusting.",
    'country': 'Colombia', 'year': '2024'
}

fiscal_stance_sample = {
    'staff': "The fiscal deficit has increased significantly. Staff recommends reducing government expenditure by 2% of GDP.",
    'authority': "The government will continue supportive fiscal policies. Additional infrastructure spending is planned.",
    'country': 'South Africa', 'year': '2024'
}

fiscal_agreement_sample = {
    'staff': "Staff recommends implementing comprehensive tax reforms to broaden the tax base and improve revenue mobilization.",
    'authority': "The government has prioritized improving public expenditure efficiency rather than increasing tax burdens.",
    'country': 'Peru', 'year': '2024'
}

print("Sample data loaded")

Sample data loaded


In [28]:
# Generic test function - simplified using PROMPT_REGISTRY
def test_prompt(prompt_key, prompt_dir, sample_data):
    """Generic function to test any prompt type using PROMPT_REGISTRY"""
    print(f"\n=== {prompt_key} ===")
    
    # Get prompt file and response model from registry
    registry_entry = PROMPT_REGISTRY[prompt_key]
    prompt_file = registry_entry["prompt_file"]
    response_model = registry_entry["response_model"]
    
    # Load and format prompt
    prompt_template = load_prompt(prompt_dir / prompt_file).sections
    formatted_template = prompt_template.copy()
    
    # Convert sample_data keys to uppercase for template formatting
    uppercase_data = {k.upper(): v for k, v in sample_data.items()}
    formatted_template["user"] = formatted_template["user"].format(**uppercase_data)
    
    # Handle TEXT_AUTHOR in system prompt if needed
    if '{TEXT_AUTHOR}' in formatted_template.get("system", ""):
        formatted_template["system"] = formatted_template["system"].replace(
            '{TEXT_AUTHOR}', uppercase_data.get('TEXT_AUTHOR', 'IMF staff')
        )
    
    messages = format_messages(formatted_template, add_schema=True)
    
    try:
        response = llm_agent.get_response_content(messages, reasoning_effort='low', response_format=response_model)
    except Exception as e:
        print(f"Error: {e}")
    
    return response

# Test monetary stance (one variant)
test_prompt(prompts['monetary_stance'][0], Path('../../src/Traction/prompts'), monetary_sample)


=== monetary_stance_simple ===


MonetaryStanceResponse(stance_current='accommodative', stance_future='tightening bias')

## Summary

Demonstrates 16 fiscal and monetary policy classification prompts (4 categories × 4 variants each):
- Monetary/Fiscal Stance & Agreement Classification
- Variants: simple, with_definitions, chain_of_thought, few_shot

The generic `test_prompt()` function can test any prompt variant with appropriate sample data and response models.